# Лабораторная работа №4. EM-алгоритм

В ноутбуке проводится эксперимент для Gaussian Mixture Model: собственная реализация EM сравнивается с `sklearn.mixture.GaussianMixture` на датасете Wine.

## Краткое описание алгоритма

GMM задает плотность как смесь многомерных нормальных распределений. На E-шаге вычисляются ответственности компонент для каждого объекта, на M-шаге по этим ответственностям обновляются веса, средние и ковариации. Критерий качества для задачи восстановления плотности - логарифм правдоподобия выборки.

## Датасет

Используется Wine recognition dataset из `sklearn.datasets.load_wine`: 178 объектов, 13 числовых признаков и 3 сорта вина. Метки классов не используются при обучении, но помогают дополнительно оценить согласованность найденных компонент.

In [7]:
from pathlib import Path
import sys

import pandas as pd

LAB_DIR = Path.cwd()
if LAB_DIR.name != "lab4":
    LAB_DIR = Path("students/mukhomediarova-ar/lab4").resolve()

SOURCE_DIR = LAB_DIR / "source"
ARTIFACTS_DIR = LAB_DIR / "artifacts"
sys.path.insert(0, str(SOURCE_DIR))

from main import MODEL_PARAMS, run_experiment

MODEL_PARAMS

{'n_components': 3,
 'tol': 0.0001,
 'reg_covar': 0.0001,
 'max_iter': 300,
 'n_init': 5,
 'random_state': 42}

In [8]:
summary = run_experiment()
summary["dataset"], summary["sizes"]

('Wine recognition dataset, sklearn.datasets.load_wine',
 {'train': 124, 'test': 54, 'features': 13, 'classes': 3})

## Сравнение с sklearn

Основная метрика - log-likelihood: чем выше значение, тем лучше модель описывает плотность данных. AIC и BIC учитывают сложность модели, меньшие значения предпочтительнее.

In [9]:
metrics = pd.read_csv(ARTIFACTS_DIR / "metrics.csv")
metrics.round(4)

,model,train_log_likelihood,test_log_likelihood,train_avg_log_likelihood,test_avg_log_likelihood,test_bic,test_aic,ari_vs_true_labels,nmi_vs_true_labels,cluster_accuracy_vs_true_labels,n_iter,converged,train_time_seconds
0,Custom GaussianMixtureEM,-1400.4412,-891.8653,-11.2939,-16.5160,3036.2715,2411.7305,0.2267,0.2884,NaN,41,True,0.0657
1,sklearn GaussianMixture,-1291.3838,-810.5234,-10.4144,-15.0097,2873.5877,2249.0467,0.4500,0.5847,NaN,6,True,0.0311


In [10]:
curve = pd.read_csv(ARTIFACTS_DIR / "log_likelihood_curve.csv")
is_monotonic = curve["custom_train_avg_log_likelihood"].diff().dropna().ge(-1e-10).all()
print(f"EM converged monotonically: {is_monotonic}")
curve.tail(10).round(6)

EM converged monotonically: True


,iteration,custom_train_avg_log_likelihood
31,32,-11.309964
32,33,-11.305173
33,34,-11.300622
34,35,-11.298747
35,36,-11.297598
36,37,-11.296191
37,38,-11.294883
38,39,-11.294172
39,40,-11.293940
40,41,-11.293880


## Компоненты смеси

Ниже показаны веса компонент и первые несколько средних в исходном масштабе признаков. Порядок компонент у GMM произвольный, поэтому прямое совпадение номеров компонент не требуется.

In [11]:
component_summary = pd.read_csv(ARTIFACTS_DIR / "component_summary.csv")
columns = ["model", "component", "weight", "alcohol", "malic_acid", "ash", "alcalinity_of_ash"]
component_summary[columns].round(4)

,model,component,weight,alcohol,malic_acid,ash,alcalinity_of_ash
0,Custom GaussianMixtureEM,0,0.4186,12.4797,2.3415,2.2955,20.9522
1,Custom GaussianMixtureEM,1,0.0726,12.3611,2.1656,2.0922,18.1778
2,Custom GaussianMixtureEM,2,0.5088,13.4326,2.2595,2.4581,18.8250
3,sklearn GaussianMixture,0,0.0484,12.2117,1.8717,2.1767,20.2667
4,sklearn GaussianMixture,1,0.6297,13.0147,1.9575,2.3422,18.6542
5,sklearn GaussianMixture,2,0.3219,12.9528,2.9940,2.4332,21.5626


In [12]:
predictions = pd.read_csv(ARTIFACTS_DIR / "predictions.csv")
pd.crosstab(predictions["y_true"], predictions["custom_component"], rownames=["true class"], colnames=["custom component"])

custom component,0,2
true class,,
0,0,18
1,12,9
2,11,4


## Графики и выводы

### Сходимость EM-алгоритма

![Сходимость EM](images/em_convergence.svg)

**Вывод.** Средний log-likelihood на обучающей выборке монотонно растет и выходит на плато. Это подтверждает корректное поведение EM: после каждого E/M-шага параметры смеси не ухудшают правдоподобие, а остановка происходит, когда прирост становится малым.

### Сравнение правдоподобия

![Сравнение правдоподобия](images/likelihood_comparison.svg)

**Вывод.** Более высокий средний log-likelihood означает лучшее восстановление плотности. Если `sklearn GaussianMixture` показывает более высокое значение, это ожидаемо из-за более развитой инициализации и оптимизаций, при этом собственная реализация остается сопоставимой по постановке и критерию качества.

### Соответствие классов и компонент

![Соответствие классов и компонент](images/component_matrix.svg)

**Вывод.** Матрицы показывают, сколько объектов каждого истинного класса попало в каждую компоненту смеси. Номера компонент у GMM произвольны, поэтому важна не буквальная нумерация столбцов, а концентрация значений по отдельным компонентам. У `sklearn` соответствие классам получилось более выраженным, а собственная модель частично смешала классы 1 и 2; это допустимо, так как EM оптимизирует плотность, а не классификационную accuracy.

## Вывод

Собственная реализация обучает GMM через EM без использования меток классов и сохраняет историю сходимости. Сравнение с `sklearn GaussianMixture` выполняется по ПМП на train/test, AIC/BIC и дополнительным кластерным метрикам. Для задачи восстановления плотности главным критерием остается log-likelihood, а ARI/NMI/cluster accuracy служат только интерпретацией найденных компонент.